In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch
from scipy.signal import convolve2d

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
s = np.load('psf.npz')
psf = s['psf']


def make_map(files, dlon=0.2, dsine=1 / 359.5, stonyhurst=False, mu_thr=0.1, binning=1):
    grid = np.mgrid[-1:1 + dsine / 2:dsine, :360:dlon]
    grid[0] = np.arcsin(grid[0].clip(-1,1)) * 180 / np.pi

    lat = np.arange(-1,1 + dsine / 2, dsine)
    lat = np.arcsin(lat.clip(-1,1)) * 180 / np.pi
    lon = np.arange(0,360,dlon)


    coverage, mean, variance, w, w2 = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        print(file)

        with fits.open(file) as hdul:
            header = hdul[1].header.copy()
            data = hdul[1].data.copy()

        data = rebin(data, binning, update_header=header)
        data = convolve2d(data, psf, mode='same', boundary='symm')

        view = View.from_header(header)
        transform = ~ToSpherical() - view.to_synoptic(correct_mu=True, mu_thr=mu_thr, stonyhurst=stonyhurst)
        #transform = ~ToSpherical() - view.to_carrington(correct_mu=True, mu_thr=mu_thr, stonyhurst=stonyhurst)

        grid_, alpha = transform(grid)
        data = interp2d(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4

        coverage += ~np.isnan(data)
        w += np.nan_to_num(n)
        w2 += np.nan_to_num(n ** 2)
        mean_ = mean.copy()
        mean += np.nan_to_num((data - mean) * n / w)
        variance += np.nan_to_num((data - mean) * (data - mean_) * n)

    variance = variance / (1 - w2 / w ** 2) * w2 / w ** 3
    mean[coverage == 0] = np.nan
    variance[coverage == 0] = np.nan
    w[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    return mean, variance, coverage, w, lat, lon

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2289/*'))
print(len(files))
print(files[0], files[-1])

319
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240919_160000_TAI.3.magnetogram.fits /home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20241016_140000_TAI.3.magnetogram.fits


In [4]:
mean, variance, coverage, weight, lat, lon = make_map(files, binning=5)

/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240919_160000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240919_180000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240919_200000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240919_220000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240920_000000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240920_020000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240920_040000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240920_060000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240920_080000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240920_100000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240920_120000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2289/hmi.M_720s.20240920_140000_TAI.3.magnetogram.fits
/home/ulyanov/da

/tmp/ipykernel_49014/148181902.py:42: RuntimeWarning: invalid value encountered in divide
  variance = variance / (1 - w2 / w ** 2) * w2 / w ** 3
/tmp/ipykernel_49014/148181902.py:42: RuntimeWarning: divide by zero encountered in divide
  variance = variance / (1 - w2 / w ** 2) * w2 / w ** 3


In [5]:
fig, ax = plt.subplots(figsize=(14,8))
ax.imshow(mean, aspect='auto', origin='lower', cmap='hmimag',
           extent=(0, 360, -1, 1), vmin=-1000, vmax=1000)
ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))
plt.tight_layout()

In [6]:
np.savez('2289hmi_conv.npz', mean=mean, variance=variance, coverage=coverage, weight=weight, lat=lat, lon=lon)